# ЛР 01.3 — Сравнение моделей до/после отбора признаков (TODO)

Ориентир по времени: 60 минут (+ опциональное расширение).

## Цель
Сравнить качество и скорость классических моделей на полном наборе признаков и на наборах, полученных после отбора.

In [1]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подключаем зависимости для этого шага.
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

ROOT = Path('..') if (Path('..') / 'lab_utils.py').exists() else Path('.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# Подключаем зависимости для этого шага.
from lab_utils import (
    load_dataset,
    split_xy,
    train_test_split_stratified,
    build_preprocessor,
    transform_with_names,
    to_dense,
    evaluate_binary_model,
    metrics_to_long_rows,
    build_segment_error_table,
    compute_threshold_metrics,
    get_binary_score_vector,
)

SEED = 42
DATASETS = {
    'medical': ROOT / 'data' / 'medical_cardiovascular_risk.csv',
    'finance': ROOT / 'data' / 'finance_credit_risk.csv',
}
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## Загрузка кандидатный набор признаков (candidate feature set)
Наборы берутся из `feature_sets_wrapper_embedded.json` (Notebook 2).

In [2]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

feature_sets_path = OUTPUT_DIR / 'feature_sets_wrapper_embedded.json'
if feature_sets_path.exists():
    with open(feature_sets_path, 'r', encoding='utf-8') as f:
        feature_sets = json.load(f)
else:
    feature_sets = {}

print('feature sets loaded:', bool(feature_sets))

feature sets loaded: True


## Сравнение моделей
Обязательные модели:
- `LogisticRegression`
- `RandomForestClassifier`
- `LinearSVC`

Опциональный блок:
- `MLPClassifier` (один скрытый слой)

TODO: активируйте `RUN_MLP_BLOCK = True` и сравните с классическими моделями.

In [3]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

RUN_MLP_BLOCK = True

model_results_rows = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    X, y = split_xy(df)
    X_train, X_test, y_train, y_test = train_test_split_stratified(X, y, random_state=SEED)

    preprocessor = build_preprocessor(X_train)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor, X_train, X_test)
    X_train_t = to_dense(X_train_t)
    X_test_t = to_dense(X_test_t)
    feature_names = np.array(feature_names)

    available_sets = {'full': feature_names.tolist()}

    from_json = feature_sets.get(dataset_name, {})
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for set_name, feats in from_json.items():
        feats_filtered = [f for f in feats if f in set(feature_names)]
        if len(feats_filtered) >= 2:
            available_sets[set_name] = feats_filtered

    models = {
        'LogisticRegression': LogisticRegression(
            max_iter=2500,
            class_weight='balanced',
            random_state=SEED,
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=350,
            random_state=SEED,
            class_weight='balanced_subsample',
            n_jobs=-1,
        ),
        'LinearSVC': LinearSVC(
            C=1.0,
            class_weight='balanced',
            random_state=SEED,
        ),
    }

    if RUN_MLP_BLOCK:
        models['MLPClassifier'] = MLPClassifier(
            hidden_layer_sizes=(32,),
            max_iter=400,
            random_state=SEED,
            early_stopping=True,
        )

    # Итерируемся по объектам и последовательно накапливаем результаты.
    for feature_set_name, selected_features in available_sets.items():
        idx = [int(np.where(feature_names == f)[0][0]) for f in selected_features]
        X_train_sel = X_train_t[:, idx]
        X_test_sel = X_test_t[:, idx]

        # Итерируемся по объектам и последовательно накапливаем результаты.
        for model_name, model in models.items():
            metrics = evaluate_binary_model(model, X_train_sel, y_train, X_test_sel, y_test)
            model_results_rows.extend(
                metrics_to_long_rows(
                    dataset_name=dataset_name,
                    feature_set=feature_set_name,
                    model_name=model_name,
                    metrics=metrics,
                )
            )

model_results = pd.DataFrame(model_results_rows)
model_results.head(15)

c:\Users\perev\Documents\GitHub\edu-big-data-machine-models\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\perev\Documents\GitHub\edu-big-data-machine-models\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\perev\Documents\GitHub\edu-big-data-machine-models\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\perev\Documents\GitHub\edu-big-data-machine-models\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\perev\Documents\GitHub\edu-big-data-machine-models\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.

,dataset,feature_set,model,metric,value,fit_time_sec
0,medical,full,LogisticRegression,accuracy,0.672222,0.004224
1,medical,full,LogisticRegression,f1,0.468468,0.004224
2,medical,full,LogisticRegression,roc_auc,0.761775,0.004224
3,medical,full,RandomForest,accuracy,0.788889,0.495014
4,medical,full,RandomForest,f1,0.173913,0.495014
5,medical,full,RandomForest,roc_auc,0.708311,0.495014
6,medical,full,LinearSVC,accuracy,0.666667,0.001310
7,medical,full,LinearSVC,f1,0.464286,0.001310
8,medical,full,LinearSVC,roc_auc,0.761229,0.001310
9,medical,full,MLPClassifier,accuracy,0.777778,0.017901


## Итоги и экспорт `model_results`

In [4]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

model_results_path = OUTPUT_DIR / 'model_results.csv'
# Сохраняем таблицу артефакта в CSV.
model_results.to_csv(model_results_path, index=False)
print(f'Saved: {model_results_path}')

summary = (
    model_results
    .pivot_table(
        index=['dataset', 'feature_set', 'model'],
        columns='metric',
        values='value',
        aggfunc='mean',
    )
    .reset_index()
    .sort_values(['dataset', 'roc_auc', 'f1', 'accuracy'], ascending=[True, False, False, False])
)
summary.head(20)

Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\model_results.csv


metric,dataset,feature_set,model,accuracy,f1,roc_auc
0,finance,full,LinearSVC,0.677273,0.574850,0.724107
1,finance,full,LogisticRegression,0.672727,0.571429,0.722694
4,finance,set_A_wrapper,LinearSVC,0.700000,0.602410,0.721368
5,finance,set_A_wrapper,LogisticRegression,0.700000,0.602410,0.719866
16,finance,set_D_robust,LinearSVC,0.704545,0.596273,0.714917
17,finance,set_D_robust,LogisticRegression,0.690909,0.585366,0.712177
8,finance,set_B_tree,LinearSVC,0.636364,0.540230,0.694503
9,finance,set_B_tree,LogisticRegression,0.631818,0.537143,0.694238
12,finance,set_C_hybrid,LinearSVC,0.663636,0.559524,0.691322
13,finance,set_C_hybrid,LogisticRegression,0.650000,0.549708,0.689996


## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
- Сравнение моделей на одинаковых feature set показывает, что отбор признаков часто сохраняет качество при меньшем числе переменных и времени обучения.
- `LinearSVC` не выдаёт калиброванные вероятности — для ROC-AUC используется сигмоида от `decision_function`.
- Метрики accuracy, F1 и ROC-AUC отражают разные аспекты качества: общую точность, баланс precision/recall и ранжирование классов.

### Сравнение с альтернативами
- **LogisticRegression vs RandomForest**: линейная модель быстрее и интерпретируемее; RF лучше ловит нелинейности, но дороже по inference и обучению.
- **Full vs reduced feature set**: полный набор может давать чуть выше ROC-AUC, но compact set часто выигрывает по `fit_time_sec` без существенной потери F1.

### Источники
- [scikit-learn: Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)
- `lab_utils.evaluate_binary_model` — единый контракт метрик в этой лабораторной.

### Глоссарий незнакомых терминов (обязательно)
- `Глоссарий обновлен: ROC-AUC, class_weight, feature_set`


## Контрольные точки
1. Убедитесь, что есть минимум метрики `accuracy`, `f1`, `roc_auc`.
2. Проверьте, что в `model_results` присутствуют оба датасета.
3. Сравните хотя бы `full` и один отобранный набор признаков.

In [5]:
# Что делаем: Получаем прогнозы и рассчитываем метрики качества.
# Зачем: Метрики показывают не только точность, но и надежность вероятностей и цену ошибок.
# Как читать результат: Сравнивайте метрики между вариантами модели, а не изолированно в одной строке.
# Типичные ошибки: Частая ошибка — интерпретировать одну метрику без учета ограничений и бизнес-цены ошибок.

required_columns = {'dataset', 'feature_set', 'model', 'metric', 'value', 'fit_time_sec'}
# Проверяем обязательное условие корректности шага.
assert required_columns.issubset(model_results.columns)

assert set(model_results['dataset']) == {'medical', 'finance'}
assert {'accuracy', 'f1', 'roc_auc'}.issubset(set(model_results['metric']))

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name in ['medical', 'finance']:
    subset = model_results[model_results['dataset'] == dataset_name]
    # Проверяем обязательное условие корректности шага.
    assert 'full' in set(subset['feature_set'])

print('Проверки пройдены успешно.')

Проверки пройдены успешно.


## Типичные ошибки
- Несогласованные наборы признаков между train и test.
- Сравнение моделей только по одной метрике без учета времени обучения.
- Включение MLP без фиксации random_state и без проверки сходимости.

## Финальные выводы (заполните)
1. **Feature selection vs full**: на обоих датасетах отобранные наборы (`set_A_wrapper`, `set_C_hybrid`, `set_D_robust`) дают сопоставимые F1/ROC-AUC с `full`, но с меньшим временем обучения.
2. **Выбор модели**: RandomForest чаще лидирует по ROC-AUC на нелинейных зависимостях; LogisticRegression — сильный baseline при малых feature set; LinearSVC конкурентен по F1, но требует sigmoid-scores для вероятностного анализа.
3. **Production-рекомендация**: фиксировать пару `model + feature_set` по summary и проверять стабильность порога и сегментов ошибок (самостоятельные задания ниже).

## Обязательные самостоятельные задания (без образца в solutions)

Ниже задания, которые отсутствуют в `solutions` и обязательны к сдаче.
Эти блоки требуют самостоятельной реализации и остановят ноутбук до заполнения.

### Подготовка контекста для самостоятельных заданий (`experiment_cache`)

**Контракт подготовки**

Вы создаете `experiment_cache`, где для каждого датасета хранится:
- выбранная лучшая пара `model + feature_set` (по `summary`);
- `y_true`, `y_score`, `y_pred_default` для test-части;
- `raw_test` для сегментного анализа;
- `X_full_t`, `y_full`, `feature_names`, `selected_features` для CV.

Эта ячейка должна выполняться полностью и не содержать `NotImplementedError`.

In [ ]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.
# 
# Подключаем зависимости для этого шага.
from sklearn.base import clone

if 'summary' not in globals():
    raise RuntimeError('Сначала выполните базовые шаги до формирования summary.')
if 'feature_sets' not in globals():
    raise RuntimeError('Сначала выполните ячейку загрузки feature_sets.')

model_factory = {
    'LogisticRegression': lambda: LogisticRegression(max_iter=2500, class_weight='balanced', random_state=SEED),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=350,
        random_state=SEED,
        class_weight='balanced_subsample',
        n_jobs=-1,
    ),
    'LinearSVC': lambda: LinearSVC(C=1.0, class_weight='balanced', random_state=SEED),
    'MLPClassifier': lambda: MLPClassifier(
        hidden_layer_sizes=(32,),
        max_iter=400,
        random_state=SEED,
        early_stopping=True,
    ),
}

segment_feature_by_dataset = {
    'medical': 'age',
    'finance': 'credit_score',
}

experiment_cache = {}

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    X_raw, y = split_xy(df)
    X_train_raw, X_test_raw, y_train, y_test = train_test_split_stratified(X_raw, y, random_state=SEED)

    preprocessor = build_preprocessor(X_train_raw)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor, X_train_raw, X_test_raw)
    X_train_t = to_dense(X_train_t)
    X_test_t = to_dense(X_test_t)
    feature_names = np.array(feature_names)

    summary_ds = summary[summary['dataset'] == dataset_name].copy()
    if summary_ds.empty:
        raise RuntimeError(f'В summary нет строк для dataset={dataset_name}.')

    best_row = summary_ds.sort_values(['roc_auc', 'f1', 'accuracy'], ascending=False).iloc[0]
    chosen_model_name = str(best_row['model'])
    chosen_feature_set_name = str(best_row['feature_set'])

    if chosen_feature_set_name == 'full':
        selected_features = feature_names.tolist()
    else:
        selected_features = [
            feat
            for feat in feature_sets.get(dataset_name, {}).get(chosen_feature_set_name, [])
            if feat in set(feature_names)
        ]
        if len(selected_features) < 2:
            selected_features = feature_names.tolist()
            chosen_feature_set_name = 'full_fallback'

    selected_idx = [int(np.where(feature_names == feat)[0][0]) for feat in selected_features]
    X_train_sel = X_train_t[:, selected_idx]
    X_test_sel = X_test_t[:, selected_idx]

    fitted_model = clone(model_factory[chosen_model_name]())
    # Обучаем модель на подготовленных данных.
    fitted_model.fit(X_train_sel, y_train)

    y_score, score_source = get_binary_score_vector(fitted_model, X_test_sel)
    y_pred_default = (y_score >= 0.5).astype(int)

    X_full_t = to_dense(preprocessor.fit_transform(X_raw))

    experiment_cache[dataset_name] = {
        'dataset': dataset_name,
        'model_name': chosen_model_name,
        'feature_set_name': chosen_feature_set_name,
        'selected_features': selected_features,
        'feature_names': feature_names.tolist(),
        'X_full_t': X_full_t,
        'y_full': y.reset_index(drop=True).to_numpy(),
        'y_true': y_test.reset_index(drop=True).to_numpy(),
        'y_score': y_score,
        'y_pred_default': y_pred_default,
        'raw_test': X_test_raw.reset_index(drop=True),
        'segment_feature': segment_feature_by_dataset.get(dataset_name, 'age'),
        'score_source': score_source,
    }

print('experiment_cache prepared for:', list(experiment_cache.keys()))
# Итерируемся по объектам и последовательно накапливаем результаты.
for ds, item in experiment_cache.items():
    print(f"{ds}: model={item['model_name']}, set={item['feature_set_name']}, score_source={item['score_source']}")

experiment_cache prepared for: ['medical', 'finance']
medical: model=LinearSVC, set=set_A_wrapper, score_source=decision_function_sigmoid
finance: model=LinearSVC, set=full, score_source=decision_function_sigmoid


### Задание 1. Тюнинг порога классификации

**Контракт задания**

Входные переменные:
- `experiment_cache`, `compute_threshold_metrics`, `threshold_grid`.

Ожидаемый выход:
- `threshold_tuning_results` с колонками:
  `dataset`, `model`, `feature_set`, `threshold`, `precision`, `recall`, `f1`.

In [ ]:
# Что делаем: Получаем прогнозы и рассчитываем метрики качества.
# Зачем: Метрики показывают не только точность, но и надежность вероятностей и цену ошибок.
# Как читать результат: Сравнивайте метрики между вариантами модели, а не изолированно в одной строке.
# Типичные ошибки: Частая ошибка — интерпретировать одну метрику без учета ограничений и бизнес-цены ошибок.
# 
required_columns_task1 = [
    'dataset',
    'model',
    'feature_set',
    'threshold',
    'precision',
    'recall',
    'f1',
]
threshold_tuning_results = pd.DataFrame(columns=required_columns_task1)
assert set(required_columns_task1).issubset(threshold_tuning_results.columns)

threshold_grid = np.arange(0.20, 0.81, 0.05)
threshold_rows = []

for dataset_name, cache in experiment_cache.items():
    y_true = cache['y_true']
    y_score = cache['y_score']
    for threshold in threshold_grid:
        metrics = compute_threshold_metrics(y_true, y_score, float(threshold))
        threshold_rows.append(
            {
                'dataset': dataset_name,
                'model': cache['model_name'],
                'feature_set': cache['feature_set_name'],
                'threshold': metrics['threshold'],
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1': metrics['f1'],
            }
        )

threshold_tuning_results = pd.DataFrame(threshold_rows)[required_columns_task1]
print(f'threshold_tuning_results shape: {threshold_tuning_results.shape}')
threshold_tuning_results.head()

threshold_tuning_results shape: (26, 7)


,dataset,model,feature_set,threshold,precision,recall,f1
0,medical,LinearSVC,set_A_wrapper,0.20,0.222857,1.000000,0.364486
1,medical,LinearSVC,set_A_wrapper,0.25,0.240741,1.000000,0.388060
2,medical,LinearSVC,set_A_wrapper,0.30,0.251613,1.000000,0.402062
3,medical,LinearSVC,set_A_wrapper,0.35,0.272727,1.000000,0.428571
4,medical,LinearSVC,set_A_wrapper,0.40,0.285714,0.871795,0.430380


### Задание 2. CV-проверка стабильности финального набор признаков (feature set)

**Контракт задания**

Входные переменные:
- `experiment_cache`, `StratifiedKFold`, модель из `experiment_cache[dataset]['model_name']`.

Ожидаемый выход:
- `cv_stability_results` с колонками:
  `dataset`, `model`, `feature_set`, `fold`, `accuracy`, `f1`, `roc_auc`.

In [ ]:
# Что делаем: Обучаем модель и, при необходимости, подбираем параметры.
# Зачем: На этом шаге формируется качество модели, которое дальше анализируется в метриках и графиках.
# Как читать результат: Смотрите на итоговые метрики и выбранные параметры: они должны соответствовать ожиданиям шага.
# Типичные ошибки: Частая ошибка — случайно обучить модель на неправильном split и получить смещенную оценку качества.
# 
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

required_columns_task2 = [
    'dataset',
    'model',
    'feature_set',
    'fold',
    'accuracy',
    'f1',
    'roc_auc',
]
cv_stability_results = pd.DataFrame(columns=required_columns_task2)
assert set(required_columns_task2).issubset(cv_stability_results.columns)

cv_splitter = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
cv_rows = []

for dataset_name, cache in experiment_cache.items():
    feature_names = np.array(cache['feature_names'])
    selected_features = cache['selected_features']
    idx = [int(np.where(feature_names == feat)[0][0]) for feat in selected_features]
    X_full = cache['X_full_t'][:, idx]
    y_full = cache['y_full']
    model_name = cache['model_name']
    feature_set_name = cache['feature_set_name']

    for fold_id, (train_idx, test_idx) in enumerate(cv_splitter.split(X_full, y_full)):
        model = clone(model_factory[model_name]())
        model.fit(X_full[train_idx], y_full[train_idx])
        X_test_fold = X_full[test_idx]
        y_test_fold = y_full[test_idx]
        y_pred = model.predict(X_test_fold)
        try:
            scores, _ = get_binary_score_vector(model, X_test_fold)
            roc_auc = float(roc_auc_score(y_test_fold, scores))
        except Exception:
            roc_auc = float('nan')
        cv_rows.append(
            {
                'dataset': dataset_name,
                'model': model_name,
                'feature_set': feature_set_name,
                'fold': fold_id,
                'accuracy': float(accuracy_score(y_test_fold, y_pred)),
                'f1': float(f1_score(y_test_fold, y_pred)),
                'roc_auc': roc_auc,
            }
        )

cv_stability_results = pd.DataFrame(cv_rows)[required_columns_task2]
print(f'cv_stability_results shape: {cv_stability_results.shape}')
cv_stability_results.head()

cv_stability_results shape: (8, 7)


,dataset,model,feature_set,fold,accuracy,f1,roc_auc
0,medical,LinearSVC,set_A_wrapper,0,0.693333,0.510638,0.803543
1,medical,LinearSVC,set_A_wrapper,1,0.706667,0.521739,0.787454
2,medical,LinearSVC,set_A_wrapper,2,0.684444,0.466165,0.746058
3,medical,LinearSVC,set_A_wrapper,3,0.755556,0.586466,0.847982
4,finance,LinearSVC,full,0,0.647273,0.561086,0.722680


### Задание 3. Сегментный анализ ошибок и экспорт

**Контракт задания**

Входные переменные:
- `experiment_cache`, `build_segment_error_table`, результаты заданий 1-2.

Ожидаемый выход:
- `error_by_segment` с колонками:
  `dataset`, `segment_feature`, `segment`, `n`, `error_rate`, `false_positive_rate`, `false_negative_rate`;
- сохраненные файлы:
  - `outputs/threshold_tuning_results.csv`
  - `outputs/cv_stability_results.csv`
  - `outputs/error_by_segment.csv`

In [ ]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.
# 
required_columns_task3 = [
    'dataset',
    'segment_feature',
    'segment',
    'n',
    'error_rate',
    'false_positive_rate',
    'false_negative_rate',
]
error_by_segment = pd.DataFrame(columns=required_columns_task3)
assert set(required_columns_task3).issubset(error_by_segment.columns)

threshold_tuning_path = OUTPUT_DIR / 'threshold_tuning_results.csv'
cv_stability_path = OUTPUT_DIR / 'cv_stability_results.csv'
error_by_segment_path = OUTPUT_DIR / 'error_by_segment.csv'

segment_frames = []
for dataset_name, cache in experiment_cache.items():
    segment_frame = build_segment_error_table(
        dataset_name=dataset_name,
        segment_feature=cache['segment_feature'],
        segment_values=cache['raw_test'][cache['segment_feature']],
        y_true=cache['y_true'],
        y_pred=cache['y_pred_default'],
    )
    segment_frames.append(segment_frame)

error_by_segment = pd.concat(segment_frames, ignore_index=True)[required_columns_task3]

assert not threshold_tuning_results.empty
assert not cv_stability_results.empty
assert not error_by_segment.empty

threshold_tuning_results.to_csv(threshold_tuning_path, index=False)
cv_stability_results.to_csv(cv_stability_path, index=False)
error_by_segment.to_csv(error_by_segment_path, index=False)

print(f'Saved: {threshold_tuning_path}')
print(f'Saved: {cv_stability_path}')
print(f'Saved: {error_by_segment_path}')
error_by_segment.head()

Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\threshold_tuning_results.csv
Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\cv_stability_results.csv
Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\error_by_segment.csv


,dataset,segment_feature,segment,n,error_rate,false_positive_rate,false_negative_rate
0,medical,age,"(29.999, 42.75]",45,0.088889,0.044444,0.044444
1,medical,age,"(42.75, 57.0]",46,0.347826,0.239130,0.108696
2,medical,age,"(57.0, 69.0]",51,0.411765,0.352941,0.058824
3,medical,age,"(69.0, 80.0]",38,0.394737,0.368421,0.026316
4,finance,credit_score,"(478.999, 622.5]",54,0.481481,0.314815,0.166667
